# 01 unzip and stage RESP-303

Executed notebook captured from the Databricks workspace run (outputs recorded from the run).
Outputs are the notebook's own printed output; source is the notebook as run.


# 01 - Unzip and stage an approved study package

Lakeflow Declarative Pipelines ingest CSV/JSON natively but cannot open zip archives or Excel
workbooks, so this task:
1. Validates the approved package (`study_metadata.json` present, study id matches).
2. Extracts each file into `raw/<study_id>/<domain>/<submission_id>__<file>.csv`.
3. Converts vendor Excel files to CSV with normalized headers.
4. Records the package in `study_registry` and `submission_files`.

In [ ]:
%pip install openpyxl --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

for name in ["submission_id", "study_id", "zip_path", "catalog", "schema"]:
    dbutils.widgets.text(name, "")

submission_id = dbutils.widgets.get("submission_id").strip()
study_id_param = dbutils.widgets.get("study_id").strip()
zip_path = dbutils.widgets.get("zip_path").strip()
catalog = dbutils.widgets.get("catalog").strip() or "clinical_demo"
schema = dbutils.widgets.get("schema").strip() or "study_onboarding"

if not submission_id or not zip_path:
    raise ValueError("submission_id and zip_path are required job parameters")

raw_root = Path(f"/Volumes/{catalog}/{schema}/raw")
print(f"submission={submission_id} study={study_id_param or '<from metadata>'} zip={zip_path}")

submission=SUB-20261125171513-IYLA study=RESP-303 zip=/Volumes/<catalog>/study_onboarding/landing/SUB-20261125171513-IYLA__RESP-303_study_package.zip


In [ ]:
DOMAIN_BY_STEM = {
    "demographics": "dm",
    "adverse_events": "ae",
    "vital_signs": "vs",
    "exposure": "ex",
    "disposition": "ds",
    "site_visits": "sv",
    "central_lab_results": "lb",
    "labs": "lb",
}


def normalize_header(col: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "_", str(col).strip().upper()).strip("_")


with open(zip_path, "rb") as fh:
    archive = zipfile.ZipFile(io.BytesIO(fh.read()))

members = [m for m in archive.infolist() if not m.is_dir() and not Path(m.filename).name.startswith(".")]
meta_member = next((m for m in members if Path(m.filename).name == "study_metadata.json"), None)
if meta_member is None:
    raise ValueError("Package rejected: study_metadata.json is missing")

metadata = json.loads(archive.read(meta_member))
study_id = metadata.get("study_id", "").strip()
if study_id_param and study_id_param != study_id:
    raise ValueError(f"Package study_id '{study_id}' does not match submission study_id '{study_id_param}'")
declared = {Path(k).stem: v.get("domain") for k, v in metadata.get("files", {}).items()}

In [ ]:
staged = []
for m in members:
    src = Path(m.filename)
    stem, suffix = src.stem.lower(), src.suffix.lower()
    if src.name == "study_metadata.json":
        target = raw_root / study_id / "meta" / f"{submission_id}__study_metadata.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        meta_out = {k: v for k, v in metadata.items() if k != "files"}
        meta_out.update(submission_id=submission_id, staged_at=datetime.now(timezone.utc).isoformat())
        target.write_text(json.dumps(meta_out))
        staged.append(("meta", src.as_posix(), target.as_posix(), 1, "staged"))
        continue

    domain = DOMAIN_BY_STEM.get(stem) or declared.get(src.stem)
    if not domain:
        staged.append(("unknown", src.as_posix(), None, 0, "skipped: unrecognized file"))
        continue

    target = raw_root / study_id / domain / f"{submission_id}__{stem}.csv"
    target.parent.mkdir(parents=True, exist_ok=True)
    if suffix in (".xlsx", ".xls"):
        df = pd.read_excel(io.BytesIO(archive.read(m)), dtype=str)
    elif suffix == ".csv":
        df = pd.read_csv(io.BytesIO(archive.read(m)), dtype=str, keep_default_na=False)
    else:
        staged.append((domain, src.as_posix(), None, 0, f"skipped: unsupported format {suffix}"))
        continue
    df.columns = [normalize_header(c) for c in df.columns]
    df.to_csv(target, index=False)
    staged.append((domain, src.as_posix(), target.as_posix(), len(df), "staged"))

for row in staged:
    print(row)

if not any(s[0] == "dm" and s[4] == "staged" for s in staged):
    raise ValueError("Package rejected: demographics (dm) file is required")

('dm', 'RESP-303/edc/demographics.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/dm/SUB-20261125171513-IYLA__demographics.csv', 88, 'staged')
('ae', 'RESP-303/edc/adverse_events.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/ae/SUB-20261125171513-IYLA__adverse_events.csv', 225, 'staged')
('vs', 'RESP-303/edc/vital_signs.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/vs/SUB-20261125171513-IYLA__vital_signs.csv', 2380, 'staged')
('ex', 'RESP-303/edc/exposure.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/ex/SUB-20261125171513-IYLA__exposure.csv', 88, 'staged')
('ds', 'RESP-303/edc/disposition.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/ds/SUB-20261125171513-IYLA__disposition.csv', 88, 'staged')
('sv', 'RESP-303/edc/site_visits.csv', '/Volumes/<catalog>/study_onboarding/raw/RESP-303/sv/SUB-20261125171513-IYLA__site_visits.csv', 476, 'staged')
('lb', 'RESP-303/vendor/central_lab_results.xlsx', '/Volumes/<catalog>/study_onboarding/raw/RESP-303

In [ ]:
now = datetime.now(timezone.utc)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.study_registry (
  study_id STRING, nct_id STRING, title STRING, phase STRING, indication STRING,
  therapeutic_area STRING, sponsor STRING, planned_enrollment INT,
  latest_submission_id STRING, files_staged INT, rows_staged BIGINT, staged_at TIMESTAMP
) COMMENT 'Operational registry of onboarded studies and their latest approved package'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.submission_files (
  submission_id STRING, study_id STRING, domain STRING, source_file STRING,
  staged_path STRING, row_count BIGINT, status STRING, staged_at TIMESTAMP
) COMMENT 'File-level manifest of every staged study package'
""")

files_df = spark.createDataFrame(
    [(submission_id, study_id, d, s, t, int(n), st, now) for d, s, t, n, st in staged],
    "submission_id STRING, study_id STRING, domain STRING, source_file STRING, staged_path STRING, row_count BIGINT, status STRING, staged_at TIMESTAMP",
)
files_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.submission_files")

staged_ok = [s for s in staged if s[4] == "staged" and s[0] != "meta"]
reg = spark.createDataFrame(
    [(
        study_id, metadata.get("nct_id"), metadata.get("title"), metadata.get("phase"),
        metadata.get("indication"), metadata.get("therapeutic_area"), metadata.get("sponsor"),
        int(metadata.get("planned_enrollment") or 0), submission_id,
        len(staged_ok), int(sum(s[3] for s in staged_ok)), now,
    )],
    "study_id STRING, nct_id STRING, title STRING, phase STRING, indication STRING, therapeutic_area STRING, sponsor STRING, planned_enrollment INT, latest_submission_id STRING, files_staged INT, rows_staged BIGINT, staged_at TIMESTAMP",
)
reg.createOrReplaceTempView("incoming_registry")
spark.sql(f"""
MERGE INTO {catalog}.{schema}.study_registry t USING incoming_registry s ON t.study_id = s.study_id
WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
""")

dbutils.jobs.taskValues.set("study_id", study_id)
print(f"Staged {len(staged_ok)} files for {study_id}")

Staged 7 files for RESP-303
